# Data preparation

In [57]:
# import libraries
import pandas as pd
import numpy as np


In [58]:
# loading the  fear and greed dataset
fg = pd.read_csv("data\\fear_greed_index.csv", parse_dates=["date"])
fg.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [59]:
print(f"Shape: {fg.shape}")         
print(fg.dtypes)                     
print(fg.isnull().sum())             
print(fg.duplicated().sum())         

Shape: (2644, 4)
timestamp                  int64
value                      int64
classification            object
date              datetime64[ns]
dtype: object
timestamp         0
value             0
classification    0
date              0
dtype: int64
0


In [60]:
# loading the trader dataset
trader = pd.read_csv("data\\historical_data.csv")
trader.head(2)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12


In [61]:
print(f"Shape: {trader.shape}")
print(trader.dtypes)
print(trader.isnull().sum())
print(trader.duplicated().sum())

Shape: (211224, 16)
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64
0


In [62]:
# Parse the human-readable IST timestamp
# format="%d-%m-%Y %H:%M" is explicit — never let pandas guess
# because 02-12-2024 could be read as Feb 12 (wrong) or Dec 2 (correct)
trader["Timestamp IST"] = pd.to_datetime(
    trader["Timestamp IST"],
    format="%d-%m-%Y %H:%M"
)

print(f"Timestamp IST dtype: {trader['Timestamp IST'].dtype}")
print(trader["Timestamp IST"].head(3))

Timestamp IST dtype: datetime64[ns]
0   2024-12-02 22:50:00
1   2024-12-02 22:50:00
2   2024-12-02 22:50:00
Name: Timestamp IST, dtype: datetime64[ns]


In [63]:
# Before trusting or dropping the Unix Timestamp column
# check how many unique values it has
print(f"Unique Timestamp values: {trader['Timestamp'].nunique()}")
print(trader["Timestamp"].value_counts())

Unique Timestamp values: 7
Timestamp
1.740000e+12    133871
1.730000e+12     35241
1.750000e+12     26961
1.720000e+12      7141
1.710000e+12      6962
1.700000e+12      1045
1.680000e+12         3
Name: count, dtype: int64


In [64]:
# Convert Unix timestamp (milliseconds) to IST for comparison
trader["ts_unix_check"] = pd.to_datetime(
    trader["Timestamp"], unit="ms", utc=True
).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.floor("min")

# Compare dates from both columns
trader["date_from_ist"]  = trader["Timestamp IST"].dt.date
trader["date_from_unix"] = trader["ts_unix_check"].dt.date

date_mismatch = trader["date_from_ist"] != trader["date_from_unix"]

print(f"Total rows:        {len(trader)}")
print(f"Date mismatches:   {date_mismatch.sum()}")
print(f"Mismatch %:        {date_mismatch.mean()*100:.1f}%")

print("\nDate distribution from Timestamp IST:")
print(trader["date_from_ist"].value_counts().sort_index())

print("\nDate distribution from Unix Timestamp:")
print(trader["date_from_unix"].value_counts().sort_index())

Total rows:        211224
Date mismatches:   209073
Mismatch %:        99.0%

Date distribution from Timestamp IST:
date_from_ist
2023-05-01       3
2023-12-05       9
2023-12-14      11
2023-12-15       2
2023-12-16       3
              ... 
2025-04-27     337
2025-04-28    1379
2025-04-29    2243
2025-04-30    1113
2025-05-01    1230
Name: count, Length: 480, dtype: int64

Date distribution from Unix Timestamp:
date_from_unix
2023-03-28         3
2023-11-15      1045
2024-03-09      6962
2024-07-03      7141
2024-10-27     35241
2025-02-20    133871
2025-06-15     26961
Name: count, dtype: int64


### Summarizing  what I found
`FINDING:` Unix Timestamp column is corrupted.

`Root cause:` CSV was exported via Excel/Google Sheets which
auto-formatted the 13-digit millisecond Unix timestamp
(e.g. 1,733,081,447,382) into scientific notation
(1.730000e+12), rounding away the last 9 digits and
collapsing 211,224 unique timestamps into only 7 values.

`Example:`
  Real timestamp:    1,733,081,447,382  (unique per trade)
  After Excel:       1,730,000,000,000  (all Nov-Dec trades identical)

This is irreversible the original precision cannot be recovered.

`Decision:` Drop the Unix Timestamp column entirely.
          Use Timestamp IST as the sole time reference.
          Timestamp IST shows a clean distribution across
          Oct 2024 - May 2025 with realistic daily trade counts.


In [65]:
# Drop the corrupted Unix Timestamp and all investigation columns
cols_to_drop = [c for c in ["Timestamp", "ts_unix_check",
                             "date_from_ist", "date_from_unix"]
                if c in trader.columns]

trader.drop(columns=cols_to_drop, inplace=True)

# Extract clean date column for merging with Fear/Greed data
trader["date"] = pd.to_datetime(trader["Timestamp IST"].dt.date)

print(f"Columns remaining: {len(trader.columns)}")
print(trader.columns.tolist())
print(f"\nDate range: {trader['date'].min()} to {trader['date'].max()}")
print(f"Unique trade days: {trader['date'].nunique()}")
print("\nTrades per day (sample):")
print(trader["date"].value_counts().sort_index().tail(10))

Columns remaining: 16
['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date']

Date range: 2023-05-01 00:00:00 to 2025-05-01 00:00:00
Unique trade days: 480

Trades per day (sample):
date
2025-04-22    2998
2025-04-23    6159
2025-04-24    2232
2025-04-25    1653
2025-04-26    1131
2025-04-27     337
2025-04-28    1379
2025-04-29    2243
2025-04-30    1113
2025-05-01    1230
Name: count, dtype: int64


In [66]:
# Confirm date extraction looks correct
# Should see realistic daily trade counts, no outlier dates
# Data starts May 2023, not 2024 as originally assumed

assert trader["date"].isnull().sum() == 0, \
    "Nulls found in date column"

assert trader["date"].min() >= pd.Timestamp("2023-01-01"), \
    "Dates unrealistically old — pre 2023"

assert trader["date"].max() < pd.Timestamp("2026-01-01"), \
    "Future dates found"

assert trader["date"].nunique() == 480, \
    f"Expected 480 unique dates, got {trader['date'].nunique()}"

print("Sanity checks passed:")
print(f"  No null dates")
print(f"  Min date: {trader['date'].min().date()}")
print(f"  Max date: {trader['date'].max().date()}")
print(f"  Unique trading days: {trader['date'].nunique()}")


Sanity checks passed:
  No null dates
  Min date: 2023-05-01
  Max date: 2025-05-01
  Unique trading days: 480


In [67]:
merged = pd.merge(
    trader,
    fg[["date", "value", "classification"]],
    on="date",
    how="left"
)
print(f"Trader rows before merge:  {len(trader)}")
print(f"Merged rows after merge:   {len(merged)}")
print(f"Columns after merge:       {merged.columns.tolist()}")
print(f"Merged shape: {merged.shape}")
print(merged["classification"].value_counts())
print(merged["classification"].isnull().sum())  # trades with no sentiment match
merged.head(2)

Trader rows before merge:  211224
Merged rows after merge:   211224
Columns after merge:       ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date', 'value', 'classification']
Merged shape: (211224, 18)
classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64
6


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,date,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,2024-12-02,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,2024-12-02,80.0,Extreme Greed


In [68]:
print("=== Sentiment coverage ===")
print(f"Rows with sentiment label: {merged['classification'].notnull().sum()}")
print(f"Rows without label (NaN):  {merged['classification'].isnull().sum()}")
print(f"Coverage %:                {merged['classification'].notnull().mean()*100:.2f}%")

print("\nSentiment distribution:")
dist = merged["classification"].value_counts()
dist_pct = merged["classification"].value_counts(normalize=True).mul(100).round(1)

print(pd.DataFrame({"count": dist, "percent": dist_pct}))

=== Sentiment coverage ===
Rows with sentiment label: 211218
Rows without label (NaN):  6
Coverage %:                100.00%

Sentiment distribution:
                count  percent
classification                
Fear            61837     29.3
Greed           50303     23.8
Extreme Greed   39992     18.9
Neutral         37686     17.8
Extreme Fear    21400     10.1


In [69]:
# Check which dates have missing sentiment labels
unmatched = merged[merged["classification"].isnull()]

if len(unmatched) == 0:
    print("All rows matched — 100% coverage")
else:
    print(f"{len(unmatched)} unmatched rows")
    print("\nDates with no sentiment match:")
    print(unmatched["date"].value_counts())

6 unmatched rows

Dates with no sentiment match:
date
2024-10-26    6
Name: count, dtype: int64


#### jump in the fear/greed data
`FINDING:` 6 trades on 2024-10-26 had no matching Fear/Greed entry.
Investigation showed Oct 26 is simply absent from the FG dataset
(jumps from Oct 25 to Oct 27 a known occasional gap in the index).

`Fix:` Forward-filled from Oct 25 sentiment (Greed, value=72).
This is standard practice for single-day gaps in daily indices.
Oct 27 reading was also Greed (74) confirming the fill is accurate.

Rows affected: 6 (0.003% of total)


In [70]:
# Modern syntax — replaces fillna(method="ffill")
merged["classification"] = merged["classification"].ffill()
merged["value"]          = merged["value"].ffill()

# Verify
print(f"Unmatched rows after fill: {merged['classification'].isnull().sum()}")
print(merged[merged["date"] == "2024-10-26"][
    ["date", "classification", "value"]
].drop_duplicates())

Unmatched rows after fill: 0
          date classification  value
727 2024-10-26          Greed   72.0


In [71]:
print("=== Final merged dataset ===")
print(f"Shape:            {merged.shape}")
print(f"Columns:          {merged.columns.tolist()}")
print(f"Date range:       {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Unique accounts:  {merged['Account'].nunique()}")
print(f"Unique coins:     {merged['Coin'].nunique()}")
print(f"Sentiment cover:  {merged['classification'].notnull().mean()*100:.1f}%")
print(f"\nFirst 3 rows:")
print(merged[["Account", "date", "Closed PnL", "classification", "value"]].head(3))

=== Final merged dataset ===
Shape:            (211224, 18)
Columns:          ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date', 'value', 'classification']
Date range:       2023-05-01 to 2025-05-01
Unique accounts:  32
Unique coins:     246
Sentiment cover:  100.0%

First 3 rows:
                                      Account       date  Closed PnL  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   

  classification  value  
0  Extreme Greed   80.0  
1  Extreme Greed   80.0  
2  Extreme Greed   80.0  


In [72]:
print("=== Sentiment distribution across all trades ===")
dist = merged["classification"].value_counts()
dist_pct = merged["classification"].value_counts(normalize=True).mul(100).round(1)

print(pd.DataFrame({"trade count": dist, "% of trades": dist_pct}))

print("\n======= Simplified Fear vs Greed split ========")
# Mapping all 5 categories into 3 buckets for cleaner analysis
merged["sentiment"] = merged["classification"].map({
    "Extreme Fear" : "Fear",
    "Fear"         : "Fear",
    "Neutral"      : "Neutral",
    "Greed"        : "Greed",
    "Extreme Greed": "Greed"
})

simple_dist = merged["sentiment"].value_counts()
simple_pct  = merged["sentiment"].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({"trade count": simple_dist, "% of trades": simple_pct}))

=== Sentiment distribution across all trades ===
                trade count  % of trades
classification                          
Fear                  61837         29.3
Greed                 50309         23.8
Extreme Greed         39992         18.9
Neutral               37686         17.8
Extreme Fear          21400         10.1

======= Simplified Fear vs Greed split ========
           trade count  % of trades
sentiment                          
Greed            90301         42.8
Fear             83237         39.4
Neutral          37686         17.8


#### FINDING 1 — Sentiment distribution across 211,224 trades

Contrary to expectation, Fear days (29.3%) generate more individual 
trades than any other single category, including Greed (23.8%).

Combined buckets: Greed 42.8% | Fear 39.4% | Neutral 17.8%


Key question this raises: are Fear-day trades actually profitable,
or are traders overtrading during panic?

# Analysis 

In [73]:
# ============================================================
# PART A — SECTION 6 : FEATURE ENGINEERING & EVALUATION METRICS
# ============================================================

# ------------------------------------------------------------
# Step 1 — Base helper columns on merged dataset
# ------------------------------------------------------------

# is_closed : trade actually closed (PnL != 0)
merged["is_closed"] = merged["Closed PnL"] != 0

# is_win : closed trade with positive PnL
merged["is_win_closed"] = (merged["Closed PnL"] > 0) & merged["is_closed"]

# is_long : buy side trade
merged["is_long"] = merged["Side"].str.upper() == "BUY"

print("Helper columns added:")
print(f"  is_closed      : {merged['is_closed'].sum():,} closed trades")
print(f"  is_win_closed  : {merged['is_win_closed'].sum():,} winning trades")
print(f"  is_long        : {merged['is_long'].sum():,} long trades")

# ------------------------------------------------------------
# Step 2 — Daily aggregation per account
# ------------------------------------------------------------

daily = merged.groupby(
    ["Account", "date", "classification", "sentiment", "value"]
).agg(
    # PnL metrics
    daily_pnl          = ("Closed PnL",    "sum"),

    # Volume metrics
    trade_count        = ("Closed PnL",    "count"),
    closed_trade_count = ("is_closed",     "sum"),
    total_size_usd     = ("Size USD",      "sum"),
    avg_size_usd       = ("Size USD",      "mean"),

    # Win rate on closed trades only
    win_rate           = ("is_win_closed", lambda x:
                          x.sum() / merged.loc[x.index, "is_closed"].sum()
                          if merged.loc[x.index, "is_closed"].sum() > 0
                          else np.nan),

    # Direction bias
    long_ratio         = ("is_long",       "mean"),

    # Cost
    total_fees         = ("Fee",           "sum"),
).reset_index()

print(f"\nDaily aggregation shape : {daily.shape}")
print(f"Null win rates (all open): {daily['win_rate'].isna().sum()}")

# ------------------------------------------------------------
# Step 3 — Account level summary
# ------------------------------------------------------------

account_summary = merged.groupby("Account").agg(
    total_pnl    = ("Closed PnL",  "sum"),
    total_trades = ("Closed PnL",  "count"),
    trading_days = ("date",        "nunique"),
).reset_index()

# Sort by total PnL descending
account_summary = account_summary.sort_values(
    "total_pnl", ascending=False
).reset_index(drop=True)

# Derived efficiency metrics
total_days = merged["date"].nunique()

account_summary["pnl_per_day"]     = (
    account_summary["total_pnl"] /
    account_summary["trading_days"]
).round(2)

account_summary["trades_per_day"]  = (
    account_summary["total_trades"] /
    account_summary["trading_days"]
).round(1)

account_summary["activity_rate"]   = (
    account_summary["trading_days"] / total_days * 100
).round(1)

account_summary["trade_share_pct"] = (
    account_summary["total_trades"] / len(merged) * 100
).round(1)

# Overall win rate per account
win_by_account = merged.groupby("Account").apply(
    lambda x: x["is_win_closed"].sum() / x["is_closed"].sum()
    if x["is_closed"].sum() > 0 else np.nan
).reset_index()
win_by_account.columns = ["Account", "overall_win_rate"]
account_summary = account_summary.merge(win_by_account, on="Account")

print(f"\nAccount summary shape : {account_summary.shape}")

# ------------------------------------------------------------
# Step 4 — Drawdown proxy per account per day
# ------------------------------------------------------------

# Cumulative PnL per account over time
# Drawdown = how far below the running peak at any given day
daily = daily.sort_values(["Account", "date"]).reset_index(drop=True)

daily["cum_pnl"] = daily.groupby("Account")["daily_pnl"].cumsum()

daily["running_peak"] = daily.groupby("Account")["cum_pnl"].cummax()

daily["drawdown"] = daily["cum_pnl"] - daily["running_peak"]

print(f"\nDrawdown proxy added:")
print(f"  Max drawdown across all accounts: "
      f"{daily['drawdown'].min():,.2f}")
print(f"  Median drawdown: {daily['drawdown'].median():,.2f}")

# ------------------------------------------------------------
# Step 5 — Print all metric summaries
# ------------------------------------------------------------

print("\n" + "="*55)
print("DAILY METRICS SUMMARY")
print("="*55)
print(daily[[
    "daily_pnl", "trade_count", "closed_trade_count",
    "win_rate", "long_ratio", "avg_size_usd",
    "total_fees", "drawdown"
]].describe().round(2).to_string())

print("\n" + "="*55)
print("ACCOUNT METRICS SUMMARY")
print("="*55)
print(account_summary[[
    "Account", "total_pnl", "pnl_per_day", "trades_per_day",
    "activity_rate", "trade_share_pct", "overall_win_rate"
]].to_string())

print("\n" + "="*55)
print("CONCENTRATION CHECK")
print("="*55)
top5_trades = account_summary["total_trades"].head(5).sum()
top5_pnl    = account_summary["total_pnl"].head(5).sum()
print(f"Top 5 accounts trade share : "
      f"{top5_trades/len(merged)*100:.1f}%")
print(f"Top 5 accounts PnL share   : "
      f"{top5_pnl/merged['Closed PnL'].sum()*100:.1f}%")
print(f"Profitable accounts        : "
      f"{(account_summary['total_pnl'] > 0).sum()} / "
      f"{len(account_summary)}")

print("\n" + "="*55)
print("SENTIMENT DISTRIBUTION")
print("="*55)
dist     = merged["classification"].value_counts()
dist_pct = merged["classification"].value_counts(
               normalize=True).mul(100).round(1)
print(pd.DataFrame({"trade count": dist, "% of trades": dist_pct}))

Helper columns added:
  is_closed      : 104,408 closed trades
  is_win_closed  : 86,869 winning trades
  is_long        : 102,696 long trades

Daily aggregation shape : (2341, 13)
Null win rates (all open): 648

Account summary shape : (32, 9)

Drawdown proxy added:
  Max drawdown across all accounts: -369,393.23
  Median drawdown: 0.00

DAILY METRICS SUMMARY
       daily_pnl  trade_count  closed_trade_count  win_rate  long_ratio  avg_size_usd  total_fees   drawdown
count    2341.00      2341.00             2341.00   1693.00     2341.00       2341.00     2341.00    2341.00
mean     4398.53        90.23               44.60      0.85        0.49       6989.52      105.02   -8965.42
std     28415.94       214.61              120.09      0.29        0.36      21538.69      461.39   27170.67
min   -358963.14         1.00                0.00      0.00        0.00          0.00       -4.85 -369393.23
25%         0.00         9.00                0.00      0.86        0.14        695.25       

C:\Users\Nirmal Choyal\AppData\Local\Temp\ipykernel_2316\3219456183.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  win_by_account = merged.groupby("Account").apply(
